# 04 · Building context and landmark relations (CPU)

**What this does:** for each building we will write an address for, collects everything a person could truthfully say about where it is.

**Two tables come out**

| Table | One row per | Holds |
|---|---|---|
| `targets.parquet` | building | its point, street, colony or complex, locality, suburb, pincode, road access point |
| `relations.parquet` | (building, nearby landmark) pair | how the two relate: one of 7 labels |

**The 7 relation labels** are the ones Google's Address Descriptors use, worked out here from map geometry:

| Label | Meaning | Rule in short |
|---|---|---|
| NEAR | somewhere within 300 m | the default |
| WITHIN | inside a named complex or campus | building point lies inside the shape |
| BESIDE | next door | same side of the same road, within ~35 m along it, no road in between |
| ACROSS_THE_ROAD | directly opposite | same road, opposite sides, within ~35 m along it |
| BEHIND | directly behind it | landmark fronts a road, building sits further back on the same side |
| DOWN_THE_ROAD | further along the same road | same road, 35 to 200 m along it |
| AROUND_THE_CORNER | one turn away | on a road that meets the landmark's road, short path via the junction |

The rules are strict on purpose: a pair gets a specific label only when the geometry clearly supports it. Everything else stays NEAR.

**Runtime:** about 10 minutes for 200,000 buildings, CPU. **Needs:** notebooks 01 and 03 (02 optional but recommended).

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/content/Navigate' if os.path.exists('/content') else '/kaggle/working/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

## 1. Load the map tables

In [ ]:
from bench import context as cx, pincode as pc, textgen as tg
G = P['gaz']
buildings = cx.load_buildings(G)
landmarks = cx.load_landmarks(G, buildings)
roads     = cx.load_roads(G)
area_poly, localities, suburbs = cx.load_areas(G)
pin_lookup, pin_method = pc.make_lookup(G)
print(f"{len(buildings):,} buildings | {len(landmarks):,} landmarks ({landmarks.source.value_counts().to_dict()}) | {len(roads):,} drivable roads")
print(f"{len(area_poly):,} named areas {area_poly.kind.value_counts().to_dict()} | {len(localities):,} localities | {len(suburbs):,} suburbs | pincode method: {pin_method}")

## 2. Choose the target buildings
- Only buildings of a sensible size (30 to 20,000 sq m) within 60 m of a drivable road.
- Buildings that already carry a **real** address in OpenStreetMap are set aside: they become the real-text test set in notebook 06 and must never be trained on.

In [ ]:
N_TARGETS = int(os.environ.get('NAV_N_TARGETS', 200_000))      # lower this for a quick trial run
real = tg.real_osm_rows(G)
pool = buildings[~buildings.osm_id.isin(real.building_id)]
targets = cx.pick_targets(pool, n_targets=N_TARGETS)
print(f"{len(real):,} buildings with a real address set aside | {len(targets):,} target buildings chosen")

## 3. Context for every target, then relations to every landmark within 300 m

In [ ]:
t = cx.build_targets(targets, roads, area_poly, localities, suburbs, pin_lookup)
print('share of targets that have each field:')
print({c: f"{t[c].notna().mean():.1%}" for c in ['bname', 'street', 'area_name', 'locality', 'suburb', 'pincode']})

rel = cx.label_relations(t, landmarks, roads)
print(f"\n{len(rel):,} (building, landmark) pairs")
print(rel.relation.value_counts().to_string())
print(f"\ntargets with at least one landmark within 300 m: {rel.building_id.nunique() / len(t):.1%}")
print(f"targets with at least one specific relation:      {rel[rel.relation != 'NEAR'].building_id.nunique() / len(t):.1%}")

## 4. Look at examples
Numbers cannot tell you whether "behind" really looks like behind. Each panel shows one pair: building (black square), landmark (ring), roads (grey, the building's own road darker). Check a few of each label by eye. If a label looks wrong often, tighten its thresholds in `bench/context.py` (the `R` dictionary at the top).

In [ ]:
import matplotlib.pyplot as plt, geopandas as gpd, shapely
lm_xy = gpd.GeoSeries(gpd.points_from_xy(landmarks.lon, landmarks.lat), crs='EPSG:4326').to_crs(cx.UTM)
lm_pos = dict(zip(landmarks.landmark_id, zip(lm_xy.x, lm_xy.y)))
tpos = t.set_index('building_id')
tree = shapely.STRtree(roads.geometry.values)
labels = ['ACROSS_THE_ROAD', 'BESIDE', 'BEHIND', 'DOWN_THE_ROAD', 'AROUND_THE_CORNER']
fig, axes = plt.subplots(len(labels), 4, figsize=(15, 3.6 * len(labels)))
for row, lab in zip(axes, labels):
    ex = rel[rel.relation == lab].sample(min(4, (rel.relation == lab).sum()), random_state=3)
    for ax, r in zip(row, ex.itertuples()):
        b = tpos.loc[r.building_id]; lx, ly = lm_pos[r.landmark_id]
        span = max(60, r.d_m * 1.2)
        for i in tree.query(shapely.box(b.x - span, b.y - span, b.x + span, b.y + span)):
            xs, ys = roads.geometry.values[i].xy
            ax.plot(xs, ys, color='black' if i == b.r_road else '0.6', lw=2.5 if i == b.r_road else 1.2)
        ax.plot(b.x, b.y, 's', color='black', ms=9); ax.plot(lx, ly, 'o', mfc='none', mec='black', ms=11, mew=2)
        ax.set_xlim(b.x - span, b.x + span); ax.set_ylim(b.y - span, b.y + span); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{lab.replace('_', ' ').lower()}  {r.d_m:.0f} m", fontsize=10)
plt.tight_layout(); plt.show()

## 5. Save

In [ ]:
B = P['bench']
t.to_parquet(f'{B}/targets.parquet', index=False)
rel.to_parquet(f'{B}/relations.parquet', index=False)
landmarks.to_parquet(f'{B}/landmarks_clean.parquet', index=False)
print('saved to', B)